## SafeDerm — 01. Data Extraction

Downloads HAM10000 from Kaggle, extracts it, merges the two image-part folders into one stable `data/raw/images/`, and discards the flattened MNIST-style CSVs we don't use.

Run this once. Everything downstream reads from `data/raw/images/` and `data/raw/HAM10000_metadata.csv`.

In [4]:
import shutil
import subprocess
import zipfile

import pandas as pd
from src.config import (
    PROJECT_ROOT,
    RAW_DATA_DIR,
    IMAGES_DIR,
    METADATA_PATH,
    KAGGLE_DATASET,
    EXPECTED_IMAGE_COUNT,
    EXPECTED_CLASSES,
)

In [ ]:
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)

### Download

Requires the Kaggle CLI and `~/.kaggle/kaggle.json`. See [Kaggle API setup](https://github.com/Kaggle/kaggle-api#api-credentials) if you haven't done this.

Skips the download if the images are already extracted — safe to re-run.

No `kaggle.json` set up yet? Download the zip manually from the [Kaggle dataset page](https://www.kaggle.com/datasets/kmader/skin-cancer-mnist-ham10000) instead and drop it straight into `data/raw/` — the extraction cell below works the same either way, it just looks for any `.zip` sitting in that folder.

In [ ]:
# Delete data/raw/images/ to force a fresh download from Kaggle.
if IMAGES_DIR.exists() and any(IMAGES_DIR.glob("*.jpg")):
    print("Images already present, skipping download.")
else:
    print("Downloading from Kaggle (~3GB, may take a few minutes)...")
    subprocess.run(
        [
            "kaggle", "datasets", "download",
            "-d", KAGGLE_DATASET,
            "-p", str(RAW_DATA_DIR),
        ],
        check=True,  # raises CalledProcessError on non-zero exit
    )
    print("Done.")

### Extract, consolidate, and clean up

Kaggle packages this as one ZIP containing two separate image folders (`HAM10000_images_part_1/`, `part_2/`) plus four flattened MNIST-style CSVs we don't need (`hmnist_*.csv`). We:

1. Extract everything
2. Merge both image folders into a single `data/raw/images/`
3. Delete the flattened CSVs and the ZIP — no reason to keep them

In [5]:
zip_path = next(RAW_DATA_DIR.glob("*.zip"), None)

if zip_path is not None:
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(RAW_DATA_DIR)
    zip_path.unlink()
    print("Extracted and deleted ZIP.")

# Merge both image part-folders into one stable images/ directory.
IMAGES_DIR.mkdir(parents=True, exist_ok=True)
part_folders = list(RAW_DATA_DIR.glob("HAM10000_images_part_*"))

for part in part_folders:
    for img in part.glob("*.jpg"):
        shutil.move(str(img), str(IMAGES_DIR / img.name))
    part.rmdir()

print(f"Consolidated into {IMAGES_DIR.relative_to(PROJECT_ROOT)} — {len(list(IMAGES_DIR.glob('*.jpg')))} images.")

# Drop the flattened MNIST-style CSVs — not used in this project.
for junk in RAW_DATA_DIR.glob("hmnist_*.csv"):
    junk.unlink()
    print(f"Removed unused file: {junk.name}")

Consolidated into data\raw\images — 10015 images.


### Verify

Sanity check — image count, class distribution, and the `lesion_id` vs `image_id` relationship. That last one is what the next notebook's leakage-safe split depends on. The asserts catch a corrupted download before anything downstream breaks silently.

In [6]:
assert IMAGES_DIR.exists(), "images/ folder missing — extraction failed."

n_images = len(list(IMAGES_DIR.glob("*.jpg")))
assert n_images == EXPECTED_IMAGE_COUNT, f"Expected {EXPECTED_IMAGE_COUNT} images, found {n_images}"

df = pd.read_csv(METADATA_PATH)
assert set(df["dx"].unique()) == EXPECTED_CLASSES, f"Unexpected classes: {set(df['dx'].unique())}"

# Cross-check IDs, not just counts — two sets can match in size and still not match in content.
image_files = {f.stem for f in IMAGES_DIR.glob("*.jpg")}
metadata_ids = set(df["image_id"])
missing_files = metadata_ids - image_files
extra_files = image_files - metadata_ids
assert not missing_files, f"{len(missing_files)} metadata rows have no matching image file"
assert not extra_files, f"{len(extra_files)} image files have no matching metadata row"

n_lesions = df["lesion_id"].nunique()
n_images_meta = df["image_id"].nunique()

print(f"Images on disk        : {n_images}")
print(f"Metadata rows          : {len(df)}")
print(f"Unique lesion_ids      : {n_lesions}")
print(f"Unique image_ids       : {n_images_meta}")
print(f"Lesions with >1 photo  : {(df['lesion_id'].value_counts() > 1).sum()}")
print("\nClass distribution:")
print(df["dx"].value_counts())

df.head()

Images on disk        : 10015
Metadata rows          : 10015
Unique lesion_ids      : 7470
Unique image_ids       : 10015
Lesions with >1 photo  : 1956

Class distribution:
dx
nv       6705
mel      1113
bkl      1099
bcc       514
akiec     327
vasc      142
df        115
Name: count, dtype: int64


,lesion_id,image_id,dx,dx_type,age,sex,localization
0,HAM_0000118,ISIC_0027419,bkl,histo,80.0,male,scalp
1,HAM_0000118,ISIC_0025030,bkl,histo,80.0,male,scalp
2,HAM_0002730,ISIC_0026769,bkl,histo,80.0,male,scalp
3,HAM_0002730,ISIC_0025661,bkl,histo,80.0,male,scalp
4,HAM_0001466,ISIC_0031633,bkl,histo,75.0,male,ear
